In [2]:
from ollama import chat
import pandas as pd
import json
import base64
import numpy as np
from collections import Counter
from pyspark.sql import SparkSession

In [3]:
with open("text-to-plot/combined/combined.json", "r") as f:
    combined = json.load(f)
with open("text-to-plot/combined/combined-without-type.json", "r") as f:
    combined_without_type = json.load(f)
with open("text-to-plot/combined/combined-with-type.json", "r") as f:
    combined_with_type = json.load(f)
with open("text-to-plot/combined/combined-code.json", "r") as f:
    combined_code = json.load(f)
with open("text-to-plot/combined/combined-golden.json", "r") as f:
    combined_golden = json.load(f)

In [4]:
grouped_golden = {}
for c in combined_golden:
    grouped_golden[c['uuid']] = c['plot_json']['data']

In [5]:
def generate(system_message: str, user_message: str, model: str):
    return chat(
        model=model, 
        messages =[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}],
        options={
            "temperature": 0
        }
    )

In [6]:
def save_json(file_name: str, data):
    with open(file_name, "w") as f:
        json.dump(data, f, indent=4) 

In [7]:
def decode_bdata(entry):
    raw = base64.b64decode(entry["bdata"])
    arr = np.frombuffer(raw, dtype=np.dtype(entry["dtype"]))
    return arr

In [8]:
def get_chart_code(content: str):
    return content.split("</chart-code>")[0].split("<chart-code>")[1]

In [9]:
def get_thinking(content: str):
    return content.split("</thinking>")[0].split("<thinking>")[1]

In [10]:
def compare_results(result, uuid):
    gt = grouped_golden[uuid]
    type_hit = 0
    hit = 0.0
    miss = 0.0

    for key in gt.keys():
        if key not in result.keys():
            miss += 1
            continue
        if key == "type":
            type_hit += gt[key] == result[key]
        if type(gt[key]) == list:
            values = result[key]
            if 'dtype' in values:
                values = decode_bdata(values)
            hit += Counter(values) == Counter(gt[key])
            miss += Counter(values) != Counter(gt[key])
        else:
            hit += result[key] == gt[key]
            miss += result[key] != gt[key]

    return {
        "hit": type_hit,
        "score": hit / (hit + miss)
    }

In [11]:
def load_examples(dataset_path):
    filename = dataset_path.split("/")[-1]

    spark = SparkSession.builder.appName("CSVExample").getOrCreate()

    df = spark.read.csv(f"text-to-plot/datasets/{filename}", header=True, inferSchema=True)

    rows = df.take(5)
    rows_as_strings = [str(row) for row in rows]
    return {
        "df": df,
        "examples": "\n".join(rows_as_strings)
    }

### WithoutChartType

In [22]:
system_message = """You are a text-to-chart generating model. Always generate the charts using python and plotly library.
An example is given for the used dataset. Never generate code that loads the dataset. It is already loaded in variable df as spark.read.csv().
Generate only one code sample. Respond only with the code. In the end call fig.to_json().
Generate the code between tags <chart-code></chart-code>.

Example:
Histogram of the distribution of national election turnouts for Central/Eastern region countries.

<chart-code>
from pyspark.sql import SparkSession
import plotly.express as px

# Start Spark session
spark = SparkSession.builder.getOrCreate()

# Assuming df is already a Spark DataFrame
# Filter for Central/Eastern region countries
df_central_eastern = df.filter(df['region'] == 'Central/Eastern')

# Aggregate data
df_agg = df_central_eastern.groupBy('country').avg('nat_turnout')

# Convert to pandas DataFrame
df_pandas = df_agg.toPandas()

# Create histogram with Plotly
fig = px.histogram(df_pandas, x='avg(nat_turnout)', nbins=50, labels={'avg(nat_turnout)': 'National Election Turnout (%)'}, 
                   title='Distribution of National Election Turnouts for Central/Eastern Region Countries')

# Display plot
fig.to_json()
</chart-code>"""

#### LLama 3.2 3B

In [ ]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.2:3b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

In [31]:
save_json("text-to-plot/results/llama-32-without.json", results)

#### LLama 3.1 8B

In [23]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.1:8b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

Finished index 0
Finished index 1
Finished index 2
Finished index 3
Finished index 4
Exception 'list' object has no attribute 'toPandas' for index 5
Finished index 6
Finished index 7
Exception 'Column' object is not callable for index 8
Finished index 9
Finished index 10
Finished index 11
Finished index 12
Finished index 13
Finished index 14
Exception An error occurred while calling o2174.sum.
: java.lang.ClassCastException: class java.util.HashMap cannot be cast to class java.lang.String (java.util.HashMap and java.lang.String are in module java.base of loader 'bootstrap')
	at scala.collection.immutable.List.map(List.scala:247)
	at scala.collection.immutable.List.map(List.scala:79)
	at org.apache.spark.sql.classic.RelationalGroupedDataset.selectNumericColumns(RelationalGroupedDataset.scala:111)
	at org.apache.spark.sql.RelationalGroupedDataset.aggregateNumericColumns(RelationalGroupedDataset.scala:63)
	at org.apache.spark.sql.RelationalGroupedDataset.sum(RelationalGroupedDataset.scala

{"ts": "2025-09-16 09:06:34.079", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 6 in cell [23]", "line": "", "fragment": "__eq__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o2242.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__eq__\" was called from\nline 6 in cell [23]\n\r\n\tat org.apache.spark.s

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__eq__" was called from
line 6 in cell [23]
 for index 18
Finished index 19


{"ts": "2025-09-16 09:06:43.578", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 7 in cell [23]", "line": "", "fragment": "__eq__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o2286.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__eq__\" was called from\nline 7 in cell [23]\n\r\n\tat org.apache.spark.s

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__eq__" was called from
line 7 in cell [23]
 for index 20


{"ts": "2025-09-16 09:06:48.189", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 6 in cell [23]", "line": "", "fragment": "__ge__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o2318.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__ge__\" was called from\nline 6 in cell [23]\n\r\n\tat org.apache.spark.s

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__ge__" was called from
line 6 in cell [23]
 for index 21
Finished index 22
Finished index 23
Finished index 24
Finished index 25
Finished index 26
Finished index 27
Finished index 28
Finished index 29
Exception Cannot accept list of column references or list of columns for both `x` and `y`. for index 30
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Average Total Payments` cannot be resolved. Did you mean one of the following? [`City, State`, `Classification`, `Definition`, `DRG`, `Hospital Referral Region Description`, `Provider City`, `Provider Id`, `Provider Name`, `Provider State`, `Provider Street Address`, `Provider Zip Code`, `Average Cover

{"ts": "2025-09-16 09:07:42.578", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor93.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o2621.count.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703;\n'Aggregate ['State], ['State, count(1) AS count#6378L]\n+- Filter (Classification#6344 = Alcohol and Drug U

Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703;
'Aggregate ['State], ['State, count(1) AS count#6378L]
+- Filter (Classification#6344 = Alcohol and Drug Use)
   +- Relation [City, State#6343,Classification#6344,Definition#6345,DRG#6346,Hospital Referral Region Description#6347,Provider City#6348,Provider Id#6349,Provider Name#6350,Provider State#6351,Provider Street Address#6352,Provider Zip Code#6353,Average Covered Charges #6354,Average Total Payments #6355,Number of Records#6356,Reimbursement Rate#6357,Total Discharges #6358,Total Payment#6359] csv
 for index 34
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`City, State`, `Classification`, `Definition`, `DRG`, `Hospital R

In [24]:
save_json("text-to-plot/results/llama-31-without.json", results)

#### Qwen 3B

In [25]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "qwen3:8b")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

Finished index 0
Finished index 1
Finished index 2
Exception unexpected indent (<string>, line 1) for index 3
Finished index 4
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `count` cannot be resolved. Did you mean one of the following? [`storenum`, `OPENDATE`, `date_super`, `conversion`, `st`, `county`, `STREETADDR`, `STRCITY`, `STRSTATE`, `ZIPCODE`, `type_store`, `LAT`, `LON`, `MONTH`, `DAY`, `YEAR`]. SQLSTATE: 42703 for index 5
Exception 'count(1)' for index 6
Exception unexpected indent (<string>, line 1) for index 7
Exception 'fig' for index 8
Finished index 9
Finished index 10
Exception unexpected indent (<string>, line 1) for index 11
Finished index 12
Exception All arguments should have the same length. The length of column argument `df[wide_variable_0]` is 1, whereas the length of previously-processed arguments ['y'] is 3 for index 13
Exception name 'col' is not defined for index 14
Finished index 15
Finished index 16
Finished

<string>:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Exception Object of type Interval is not JSON serializable for index 51
Finished index 52
Finished index 53


<string>:8: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Exception Object of type Interval is not JSON serializable for index 54
Finished index 55
18/56


In [26]:
save_json("text-to-plot/results/qwen-without.json", results)

### WithChartType

In [18]:
system_message = """You are a text-to-chart generating model. Always generate the charts using python and plotly library.
An example is given for the used dataset. Never generate code that loads the dataset. It is already loaded in variable df as spark.read.csv().
Generate only one code sample. Respond only with the code. In the end call fig.to_json().
Generate the code between tags <chart-code></chart-code>.

Example:
Histogram of the distribution of national election turnouts for Central/Eastern region countries.

<chart-code>
from pyspark.sql import SparkSession
import plotly.express as px

# Start Spark session
spark = SparkSession.builder.getOrCreate()

# Assuming df is already a Spark DataFrame
# Filter for Central/Eastern region countries
df_central_eastern = df.filter(df['region'] == 'Central/Eastern')

# Aggregate data
df_agg = df_central_eastern.groupBy('country').avg('nat_turnout')

# Convert to pandas DataFrame
df_pandas = df_agg.toPandas()

# Create histogram with Plotly
fig = px.histogram(df_pandas, x='avg(nat_turnout)', nbins=50, labels={'avg(nat_turnout)': 'National Election Turnout (%)'}, 
                   title='Distribution of National Election Turnouts for Central/Eastern Region Countries')

# Display plot
fig.to_json()
</chart-code>"""

#### LLama 3.2 3B

In [28]:
results = []
errors = 0
for index, data in enumerate(combined_with_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.2:3b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_with_type)}")

Finished index 0
Exception 'GroupedData' object is not subscriptable for index 1
Finished index 2
Finished index 3
Exception scatter() got an unexpected keyword argument 'nbinsx' for index 4
Exception 'fig' for index 5
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `reset_index` is not supported. for index 6


<string>:12: SyntaxWarning:

invalid escape sequence '\d'



Exception 'Column' object is not callable for index 7
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `resetRowId` is not supported. for index 8
Exception [PATH_NOT_FOUND] Path does not exist: file:/c:/Users/HomePC/Desktop/Docs/Python/NLP/data.csv. SQLSTATE: 42K03 for index 9
Exception name 'lag' is not defined for index 10
Exception scatter() got an unexpected keyword argument 'nbinsx' for index 11
Exception Value of 'names' is not the name of a column in 'data_frame'. Expected one of ['storenum', 'OPENDATE', 'date_super', 'conversion', 'st', 'county', 'STREETADDR', 'STRCITY', 'STRSTATE', 'ZIPCODE', 'type_store', 'LAT', 'LON', 'MONTH', 'DAY', 'YEAR'] but received: country for index 12
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `resetRowId` is not supported. for index 13
Exception scatter() got an unexpected keyword argument 'lat' for index 14
Finished index 15
Finished index 16
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `resetRowId` is not supported. for index 17
Exception [ATTRIB

In [29]:
save_json("text-to-plot/results/llama-32-with.json", results)

#### LLama 3.1 8B

In [30]:
results = []
errors = 0
for index, data in enumerate(combined_with_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.1:8b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_with_type)}")

Exception Value of 'y' is not the name of a column in 'data_frame'. Expected one of ['country', 'min(euro_turnout)'] but received: max(euro_turnout) for index 0
Finished index 1
Finished index 2
Finished index 3
Finished index 4
Finished index 5
Finished index 6
Exception 'Column' object is not callable for index 7
Finished index 8
Exception name 'pd' is not defined for index 9
Finished index 10
Finished index 11
Exception All arguments should have the same length. The length of argument `values` is 1, whereas the length of previously-processed arguments ['type_store'] is 2 for index 12
Finished index 13


{"ts": "2025-09-16 09:52:18.593", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value 'CA' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 6 in cell [30]", "line": "", "fragment": "__eq__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o7859.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'CA' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__eq__\" was called from\nline 6 in cell [30]\n\r\n\tat org.apache.spark.sql.errors.QueryE

Exception [CAST_INVALID_INPUT] The value 'CA' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__eq__" was called from
line 6 in cell [30]
 for index 14
Finished index 15
Exception module 'plotly.express' has no attribute 'arrange' for index 16
Finished index 17
Finished index 18
Finished index 19
Finished index 20
Exception Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['avg(start_lat)'] but received: min(start_lat) for index 21
Exception All arguments should have the same length. The length of argument `y` is 2, whereas the length of previously-processed arguments ['airline'] is 1 for index 22
Finished index 23
Exception No module named 'statsmodels' for index 24
Finished index 25
Finished index 26
Finished index 27
Finished index 28
Exception 'list' object has no attrib

{"ts": "2025-09-16 09:54:59.358", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 7 in cell [30]", "line": "", "fragment": "__gt__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o8819.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__gt__\" was called from\nline 7 in cell [30]\n\r\n\tat org.apache.spark.s

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__gt__" was called from
line 7 in cell [30]
 for index 57
Finished index 58
Finished index 59
Finished index 60
Finished index 61
Finished index 62
Exception name 'df_avg_citric_avg' is not defined for index 63
Finished index 64
Finished index 65
Finished index 66
Finished index 67
Finished index 68
Finished index 69
Finished index 70
Finished index 71
Finished index 72
Finished index 73
Finished index 74
Finished index 75
Finished index 76
Finished index 77
Finished index 78
Finished index 79
Finished index 80
Finished index 81
Finished index 82
Exception tuple index out of range for index 83
Finished index 84
Finished index 85
Finished index 86
Finished index 87
Finished index 88
Finished index 89

...

Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `City` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider City`, `Provider Id`]. SQLSTATE: 42703;
'Aggregate ['City], ['City, count(1) AS count#23773L]
+- Filter (Provider State#23746 = CA)
   +- Relation [City, State#23738,Classification#23739,Definition#23740,DRG#23741,Hospital Referral Region Description#23742,Provider City#23743,Provider Id#23744,Provider Name#23745,Provider State#23746,Provider Street Address#23747,Provider Zip Code#23748,Average Covered Charges #23749,Average Total Payments #23750,Number of Records#23751,Reimbursement Rate#23752,Total Discharges #23753,Total Payment#23754] csv
 for index 98
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Average Covered Charges` cannot be resolved. Did you mean one of the following? [`City, State`, `Classification`, `Definition`, `DR

{"ts": "2025-09-16 10:00:15.450", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `lifeboat` cannot be resolved. Did you mean one of the following? [`Age`, `Cabin`, `Sex`, `SibSp`, `Fare`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor93.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o10671.count.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `lifeboat` cannot be resolved. Did you mean one of the following? [`Age`, `Cabin`, `Sex`, `SibSp`, `Fare`]. SQLSTATE: 42703;\n'Aggregate ['lifeboat], ['lifeboat, count(1) AS count#25840L]\n+- Filter (Survived#25816 = 1)\n   +- Relation [PassengerId#25815,Survived#25816,Pclass#25817,N

Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `lifeboat` cannot be resolved. Did you mean one of the following? [`Age`, `Cabin`, `Sex`, `SibSp`, `Fare`]. SQLSTATE: 42703;
'Aggregate ['lifeboat], ['lifeboat, count(1) AS count#25840L]
+- Filter (Survived#25816 = 1)
   +- Relation [PassengerId#25815,Survived#25816,Pclass#25817,Name#25818,Sex#25819,Age#25820,SibSp#25821,Parch#25822,Ticket#25823,Fare#25824,Cabin#25825,Embarked#25826] csv
 for index 143
Finished index 144
Finished index 145
Finished index 146
Finished index 147
Finished index 148
Finished index 149
Finished index 150
Finished index 151
Finished index 152
Finished index 153
Finished index 154
Finished index 155
Finished index 156


{"ts": "2025-09-16 10:01:05.757", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `lifeboat` cannot be resolved. Did you mean one of the following? [`Age`, `Cabin`, `Sex`, `SibSp`, `Fare`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor93.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o10968.count.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `lifeboat` cannot be resolved. Did you mean one of the following? [`Age`, `Cabin`, `Sex`, `SibSp`, `Fare`]. SQLSTATE: 42703;\n'Aggregate ['lifeboat], ['lifeboat, count(1) AS count#26674L]\n+- Relation [PassengerId#26650,Survived#26651,Pclass#26652,Name#26653,Sex#26654,Age#26655,SibSp

Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `lifeboat` cannot be resolved. Did you mean one of the following? [`Age`, `Cabin`, `Sex`, `SibSp`, `Fare`]. SQLSTATE: 42703;
'Aggregate ['lifeboat], ['lifeboat, count(1) AS count#26674L]
+- Relation [PassengerId#26650,Survived#26651,Pclass#26652,Name#26653,Sex#26654,Age#26655,SibSp#26656,Parch#26657,Ticket#26658,Fare#26659,Cabin#26660,Embarked#26661] csv
 for index 157
29/158


In [31]:
save_json("text-to-plot/results/llama-31-with.json", results)

#### Qwen 3B

In [19]:
results = []
errors = 0
for index, data in enumerate(combined_with_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "qwen3:8b")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "chart_type": "",
            "hit": 0,
            "score": 0,
            "res_data": res_json
        })
        errors += 1

print(f"{errors}/{len(combined_with_type)}")

Finished index 0
Exception unexpected indent (<string>, line 1) for index 1
Finished index 2
Finished index 3
Finished index 4
Finished index 5
Finished index 6
Finished index 7
Finished index 8
Finished index 9
Exception list index out of range for index 10
Finished index 11
Finished index 12
Finished index 13
Finished index 14
Finished index 15
Exception list index out of range for index 16
Finished index 17
Finished index 18
Exception list index out of range for index 19
Finished index 20
Finished index 21
Finished index 22
Finished index 23
Finished index 24
Finished index 25
Finished index 26
Finished index 27
Finished index 28
Exception list index out of range for index 29
Finished index 30
Finished index 31
Finished index 32
Finished index 33
Exception unexpected indent (<string>, line 1) for index 34
Finished index 35
Finished index 36
Finished index 37
Finished index 38
Finished index 39
Finished index 40
Finished index 41
Finished index 42
Finished index 43
Finished index 44


{"ts": "2025-09-21 18:23:05.520", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 8 in cell [19]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o3931.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 8 in cell [19]\n\r\n\tat org.apache.spark.sql.errors.

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 8 in cell [19]
 for index 56


{"ts": "2025-09-21 18:24:37.451", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 5 in cell [19]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o3960.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 5 in cell [19]\n\r\n\tat org.apache.spark.sql.errors.

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 5 in cell [19]
 for index 57
Finished index 58
Finished index 59
Finished index 60
Exception unexpected indent (<string>, line 1) for index 61
Finished index 62
Finished index 63
Finished index 64
Finished index 65
Finished index 66
Finished index 67
Finished index 68
Exception name 'F' is not defined for index 69
Finished index 70
Finished index 71


<string>:15: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Finished index 72
Finished index 73
Exception unexpected indent (<string>, line 1) for index 74
Exception unexpected indent (<string>, line 1) for index 75
Finished index 76
Finished index 77
Finished index 78
Finished index 79
Finished index 80
Finished index 81
Finished index 82
Finished index 83
Finished index 84
Finished index 85
Finished index 86
Finished index 87
Finished index 88
Finished index 89
Exception cannot import name 'px' from 'plotly.express' (c:\Users\HomePC\Desktop\Docs\Python\NLP\.venv\Lib\site-packages\plotly\express\__init__.py) for index 90
Finished index 91
Finished index 92
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `col` is not supported. for index 93
Exception unexpected indent (<string>, line 1) for index 94
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`City, State`, `Classification`, `Definition`, `DRG`, `Hospital Referral Region Descripti

{"ts": "2025-09-21 20:02:38.993", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Lifeboat` cannot be resolved. Did you mean one of the following? [`Age`, `Cabin`, `Sex`, `SibSp`, `Fare`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor88.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o5764.count.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Lifeboat` cannot be resolved. Did you mean one of the following? [`Age`, `Cabin`, `Sex`, `SibSp`, `Fare`]. SQLSTATE: 42703;\n'Aggregate ['Lifeboat], ['Lifeboat, count(1) AS count#14249L]\n+- Relation [PassengerId#14225,Survived#14226,Pclass#14227,Name#14228,Sex#14229,Age#14230,SibSp#

Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Lifeboat` cannot be resolved. Did you mean one of the following? [`Age`, `Cabin`, `Sex`, `SibSp`, `Fare`]. SQLSTATE: 42703;
'Aggregate ['Lifeboat], ['Lifeboat, count(1) AS count#14249L]
+- Relation [PassengerId#14225,Survived#14226,Pclass#14227,Name#14228,Sex#14229,Age#14230,SibSp#14231,Parch#14232,Ticket#14233,Fare#14234,Cabin#14235,Embarked#14236] csv
 for index 157
39/158


In [20]:
save_json("text-to-plot/results/qwen-with.json", results)

### WithoutChartType + Reasoning

In [32]:
system_message = """You are a text-to-chart generating model. Always generate the charts using python and plotly library.
An example is given for the used dataset. Never generate code that loads the dataset. It is already loaded in variable df as spark.read.csv().
Generate only one code sample. Respond only with the code. In the end call fig.to_json().
Make sure to follow these steps in order and generate nothing else:
1) Give your thinking in the beginning between tags <thinking></thinking>
2) Generate the code between tags <chart-code></chart-code>.

Example:
Histogram of the distribution of national election turnouts for Central/Eastern region countries.

<thinking>
To analyze the distribution of national election turnouts in Central/Eastern European countries, I first need to select the turnout data for these countries. Since the data is numerical and continuous, a histogram is suitable to visualize how turnout values are distributed and to identify common ranges and outliers.
</thinking>

<chart-code>
from pyspark.sql import SparkSession
import plotly.express as px

# Start Spark session
spark = SparkSession.builder.getOrCreate()

# Assuming df is already a Spark DataFrame
# Filter for Central/Eastern region countries
df_central_eastern = df.filter(df['region'] == 'Central/Eastern')

# Aggregate data
df_agg = df_central_eastern.groupBy('country').avg('nat_turnout')

# Convert to pandas DataFrame
df_pandas = df_agg.toPandas()

# Create histogram with Plotly
fig = px.histogram(df_pandas, x='avg(nat_turnout)', nbins=50, labels={'avg(nat_turnout)': 'National Election Turnout (%)'}, 
                   title='Distribution of National Election Turnouts for Central/Eastern Region Countries')

# Display plot
fig.to_json()
</chart-code>"""

#### LLama 3.2 3B

In [33]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.2:3b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "reasoning": get_thinking(result.message.content),
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "model_output": result.message.content 
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

Finished index 0
Finished index 1
Finished index 2
Finished index 3
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `reset_index` is not supported. for index 4
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `sort_values` is not supported. for index 5
Finished index 6
Exception name 'col' is not defined for index 7
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `resetRowId` is not supported. for index 8
Exception Value of 'x' is not the name of a column in 'data_frame'. Expected one of [0, 1] but received: airport1 for index 9
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `cnt` cannot be resolved. Did you mean one of the following? [`airport1`, `sum(cnt)`]. SQLSTATE: 42703 for index 10
Finished index 11
Finished index 12
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `div` is not supported. for index 13
Exception Expected object of length 0, got length: 3 for index 14
Finished index 15
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `div` is 

{"ts": "2025-09-16 10:02:48.460", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 9 in cell [33]", "line": "", "fragment": "__eq__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o11496.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__eq__\" was called from\nline 9 in cell [33]\n\r\n\tat org.apache.spark.

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__eq__" was called from
line 9 in cell [33]
 for index 20


{"ts": "2025-09-16 10:02:51.471", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 10 in cell [33]", "line": "", "fragment": "isin", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o11534.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"isin\" was called from\nline 10 in cell [33]\n\r\n\tat org.apache.spark.sq

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"isin" was called from
line 10 in cell [33]
 for index 21
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `_get_object_id` is not supported. for index 22
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `_get_object_id` is not supported. for index 23
Finished index 24
Finished index 25
Finished index 26
Finished index 27
Exception [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions. for index 28
Exception [ATTRIBUTE_NOT_SUPPORTED] Attribute `sort_values` is not supported. for index 29
Exception 'Column' object is not callable for index 30
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, 

{"ts": "2025-09-16 10:03:20.281", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `country` cannot be resolved. Did you mean one of the following? [`DRG`, `Definition`, `Provider Id`, `City, State`, `Provider City`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor93.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o11765.count.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `country` cannot be resolved. Did you mean one of the following? [`DRG`, `Definition`, `Provider Id`, `City, State`, `Provider City`]. SQLSTATE: 42703;\n'Aggregate ['country], ['country, count(1) AS count#28633L]\n+- Filter (Classification#28599 = Alcohol a

Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `country` cannot be resolved. Did you mean one of the following? [`DRG`, `Definition`, `Provider Id`, `City, State`, `Provider City`]. SQLSTATE: 42703;
'Aggregate ['country], ['country, count(1) AS count#28633L]
+- Filter (Classification#28599 = Alcohol and Drug Use)
   +- Relation [City, State#28598,Classification#28599,Definition#28600,DRG#28601,Hospital Referral Region Description#28602,Provider City#28603,Provider Id#28604,Provider Name#28605,Provider State#28606,Provider Street Address#28607,Provider Zip Code#28608,Average Covered Charges #28609,Average Total Payments #28610,Number of Records#28611,Reimbursement Rate#28612,Total Discharges #28613,Total Payment#28614] csv
 for index 32
Number of providers with reimbursement rate greater than 1: 48
Exception 'int' object is not subscriptable for index 33


{"ts": "2025-09-16 10:03:25.915", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor93.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o11803.count.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703;\n'Aggregate ['State], ['State, count(1) AS count#28777L]\n+- Relation [City, State#28743,Classification#287

Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`DRG`, `City, State`, `Definition`, `Provider State`, `Provider Id`]. SQLSTATE: 42703;
'Aggregate ['State], ['State, count(1) AS count#28777L]
+- Relation [City, State#28743,Classification#28744,Definition#28745,DRG#28746,Hospital Referral Region Description#28747,Provider City#28748,Provider Id#28749,Provider Name#28750,Provider State#28751,Provider Street Address#28752,Provider Zip Code#28753,Average Covered Charges #28754,Average Total Payments #28755,Number of Records#28756,Reimbursement Rate#28757,Total Discharges #28758,Total Payment#28759] csv
 for index 34
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`City, State`, `Classification`, `Definition`, `DRG`, `Hospital Referral Region Description`, `Provider C

In [34]:
save_json("text-to-plot/results/llama-32-reasoning-without-type.json", results)

#### LLama 3.1 8B

In [35]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "llama3.1:8b-instruct-q4_K_S")
        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "reasoning": get_thinking(result.message.content),
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "model_output": result.message.content 
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

Finished index 0
Finished index 1
Finished index 2
Finished index 3
Finished index 4
Finished index 5
Finished index 6
Finished index 7
Finished index 8
Finished index 9
Finished index 10
Finished index 11
Finished index 12
Exception Value of 'values' is not the name of a column in 'data_frame'. Expected one of ['beef', 'pork', 'poultry', 'dairy'] but received: total exports for index 13
Exception An error occurred while calling o12658.avg.
: java.lang.ClassCastException: class java.util.ArrayList cannot be cast to class java.lang.String (java.util.ArrayList and java.lang.String are in module java.base of loader 'bootstrap')
	at scala.collection.immutable.List.map(List.scala:247)
	at scala.collection.immutable.List.map(List.scala:79)
	at org.apache.spark.sql.classic.RelationalGroupedDataset.selectNumericColumns(RelationalGroupedDataset.scala:111)
	at org.apache.spark.sql.RelationalGroupedDataset.aggregateNumericColumns(RelationalGroupedDataset.scala:63)
	at org.apache.spark.sql.Relatio

{"ts": "2025-09-16 10:10:43.824", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 6 in cell [35]", "line": "", "fragment": "__eq__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o12848.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__eq__\" was called from\nline 6 in cell [35]\n\r\n\tat org.apache.spark.

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__eq__" was called from
line 6 in cell [35]
 for index 20


{"ts": "2025-09-16 10:10:49.378", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 6 in cell [35]", "line": "", "fragment": "__ge__", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o12885.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"__ge__\" was called from\nline 6 in cell [35]\n\r\n\tat org.apache.spark.

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"__ge__" was called from
line 6 in cell [35]
 for index 21
Finished index 22
Finished index 23
Finished index 24
Finished index 25
Finished index 26
Finished index 27
Finished index 28
Exception No module named 'statsmodels' for index 29
Exception name 'df_south_carola_nevada' is not defined for index 30
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `State` cannot be resolved. Did you mean one of the following? [`City, State`, `Classification`, `Definition`, `DRG`, `Hospital Referral Region Description`, `Provider City`, `Provider Id`, `Provider Name`, `Provider State`, `Provider Street Address`, `Provider Zip Code`, `Average Covered Charges `, `Ave

In [36]:
save_json("text-to-plot/results/llama-31-reasoning-without-type.json", results)

#### Qwen 3B

In [37]:
results = []
errors = 0
for index, data in enumerate(combined_without_type):
    try:
        d = load_examples(data["dataset"])
        user_message = f"""{data['description']}

        Example data:
        {d['examples']}"""
        
        result = generate(system_message, user_message, "qwen3:8b")

        output = {}
        _ = exec(get_chart_code(result.message.content), {"df": d["df"]}, output)
        res_json = json.loads(output['fig'].to_json())
        res = compare_results(res_json['data'][0], data['uuid'])

        results.append({
            "index": index,
            "success": 1,
            "chart_type": res_json['data'][0]['type'],
            "expected_chart_type": grouped_golden[data['uuid']]['type'],
            "hit": res['hit'],
            "score": res['score'],
            "reasoning": get_thinking(result.message.content),
            "res_data": res_json['data'],
            "gt": grouped_golden[data['uuid']]
        })
        print(f"Finished index {index}")
    except Exception as e:
        print(f"Exception {e} for index {index}")
        results.append({
            "index": index,
            "success": 0,
            "model_output": result.message.content 
        })
        errors += 1

print(f"{errors}/{len(combined_without_type)}")

Finished index 0
Finished index 1
Finished index 2
Finished index 3
Exception list index out of range for index 4
Exception [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `count` cannot be resolved. Did you mean one of the following? [`storenum`, `OPENDATE`, `date_super`, `conversion`, `st`, `county`, `STREETADDR`, `STRCITY`, `STRSTATE`, `ZIPCODE`, `type_store`, `LAT`, `LON`, `MONTH`, `DAY`, `YEAR`]. SQLSTATE: 42703 for index 5
Exception unexpected indent (<string>, line 1) for index 6
Exception name 'col' is not defined for index 7
Finished index 8
Finished index 9
Finished index 10
Exception list index out of range for index 11
Finished index 12
Exception list index out of range for index 13
Finished index 14
Finished index 15
Finished index 16
Finished index 17
Finished index 18
Finished index 19
Finished index 20


{"ts": "2025-09-16 11:23:14.016", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 5 in cell [37]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o14093.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '1989-07-02' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 5 in cell [37]\n\r\n\tat org.apache.spark.sql.errors

Exception [CAST_INVALID_INPUT] The value '1989-07-02' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 5 in cell [37]
 for index 21
Finished index 22
Finished index 23
Finished index 24
Finished index 25
Finished index 26
Finished index 27
Finished index 28
Exception name 'col' is not defined for index 29
Finished index 30
Exception Value of 'y' is not the name of a column in 'data_frame'. Expected one of ['City, State', 'Classification', 'Definition', 'DRG', 'Hospital Referral Region Description', 'Provider City', 'Provider Id', 'Provider Name', 'Provider State', 'Provider Street Address', 'Provider Zip Code', 'Average Covered Charges ', 'Average Total Payments ', 'Number of Records', 'Reimbursement Rate', 'Total Discharges ', 'Total Payment'] but received: Average Total Paym

<string>:15: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Exception agg function failed [how->mean,dtype->object] for index 51
Finished index 52
Finished index 53


<string>:14: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Finished index 54
Exception Value of 'values' is not the name of a column in 'data_frame'. Expected one of ['Pclass', 'count'] but received: count_ for index 55
14/56


In [38]:
save_json("text-to-plot/results/qwen-reasoning-without-type.json", results)

### Results

#### WithoutChartType

In [23]:
with open("text-to-plot/results/llama-32-without.json", "r") as file:
    llama_32_without = json.load(file)

In [24]:
with open("text-to-plot/results/llama-31-without.json", "r") as file:
    llama_31_without = json.load(file)

In [25]:
with open("text-to-plot/results/qwen-without.json", "r") as file:
    qwen_without = json.load(file)

In [26]:
llama_32_without_hits = 0
llama_32_without_score = 0
for result in llama_32_without:
    llama_32_without_hits += result['hit']
    llama_32_without_score += result['score']

In [27]:
llama_31_without_hits = 0
llama_31_without_score = 0
for result in llama_31_without:
    llama_31_without_hits += result['hit']
    llama_31_without_score += result['score']

In [28]:
qwen_without_hits = 0
qwen_without_score = 0
for result in qwen_without:
    qwen_without_hits += result['hit']
    qwen_without_score += result['score']

In [29]:
print("*"*50)
print("LLaMA 3.2 Results")
print(f"LLama 3.2 Hits: {llama_32_without_hits/len(llama_32_without)}")
print(f"LLama 3.2 Score: {llama_32_without_score/(len(llama_32_without))}")

print("*"*50)
print("LLaMA 3.1 Results")
print(f"LLama 3.1 Hits: {llama_31_without_hits/len(llama_31_without)}")
print(f"LLama 3.1 Score: {llama_31_without_score/(len(llama_31_without))}")

print("*"*50)
print("Qwen Results")
print(f"Qwen Hits: {qwen_without_hits/len(qwen_without)}")
print(f"Qwen Score: {qwen_without_score/(len(qwen_without))}")

**************************************************
LLaMA 3.2 Results
LLama 3.2 Hits: 0.14285714285714285
LLama 3.2 Score: 0.14999999999999997
**************************************************
LLaMA 3.1 Results
LLama 3.1 Hits: 0.48214285714285715
LLama 3.1 Score: 0.3919642857142858
**************************************************
Qwen Results
Qwen Hits: 0.5357142857142857
Qwen Score: 0.4589285714285714


#### WithoutChartType + Reasoning

In [30]:
with open("text-to-plot/results/llama-32-reasoning-without-type.json", "r") as file:
    llama_32_reasoning_without = json.load(file)

In [31]:
with open("text-to-plot/results/llama-31-reasoning-without-type.json", "r") as file:
    llama_31_reasoning_without = json.load(file)

In [32]:
with open("text-to-plot/results/qwen-reasoning-without-type.json", "r") as file:
    qwen_reasoning_without = json.load(file)

In [33]:
llama_32_reasoning_without_hits = 0
llama_32_reasoning_without_score = 0
for result in llama_32_reasoning_without:
    try:
        llama_32_reasoning_without_hits += result['hit']
        llama_32_reasoning_without_score += result['score']
    except:
        ...

In [34]:
llama_31_reasoning_without_hits = 0
llama_31_reasoning_without_score = 0
for result in llama_31_reasoning_without:
    try:
        llama_31_reasoning_without_hits += result['hit']
        llama_31_reasoning_without_score += result['score']
    except:
        ...

In [35]:
qwen_reasoning_without_hits = 0
qwen_reasoning_without_score = 0
for result in qwen_reasoning_without:
    try:
        qwen_reasoning_without_hits += result['hit']
        qwen_reasoning_without_score += result['score']
    except:
        ...

In [36]:
print("*"*50)
print("LLaMA 3.2 Results")
print(f"LLama 3.2 Hits: {llama_32_reasoning_without_hits/len(llama_32_reasoning_without)}")
print(f"LLama 3.2 Score: {llama_32_reasoning_without_score/(len(llama_32_reasoning_without))}")

print("*"*50)
print("LLaMA 3.1 Results")
print(f"LLama 3.1 Hits: {llama_31_reasoning_without_hits/len(llama_31_reasoning_without)}")
print(f"LLama 3.1 Score: {llama_31_reasoning_without_score/(len(llama_31_reasoning_without))}")

print("*"*50)
print("Qwen Results")
print(f"Qwen Hits: {qwen_reasoning_without_hits/len(qwen_reasoning_without)}")
print(f"Qwen Score: {qwen_reasoning_without_score/(len(qwen_reasoning_without))}")

**************************************************
LLaMA 3.2 Results
LLama 3.2 Hits: 0.25
LLama 3.2 Score: 0.21428571428571425
**************************************************
LLaMA 3.1 Results
LLama 3.1 Hits: 0.44642857142857145
LLama 3.1 Score: 0.38928571428571435
**************************************************
Qwen Results
Qwen Hits: 0.5
Qwen Score: 0.41071428571428575


#### WithChartType

In [37]:
with open("text-to-plot/results/llama-32-with.json", "r") as file:
    llama_32_with = json.load(file)

In [38]:
with open("text-to-plot/results/llama-31-with.json", "r") as file:
    llama_31_with = json.load(file)

In [39]:
with open("text-to-plot/results/qwen-with.json", "r") as file:
    qwen_with = json.load(file)

In [40]:
llama_32_with_hits = 0
llama_32_with_score = 0
for result in llama_32_with:
    llama_32_with_hits += result['hit']
    llama_32_with_score += result['score']

In [41]:
llama_31_with_hits = 0
llama_31_with_score = 0
for result in llama_31_with:
    llama_31_with_hits += result['hit']
    llama_31_with_score += result['score']

In [42]:
qwen_with_hits = 0
qwen_with_score = 0
for result in qwen_with:
    qwen_with_hits += result['hit']
    qwen_with_score += result['score']

In [43]:
print("*"*50)
print("LLaMA 3.2 Results")
print(f"LLama 3.2 Hits: {llama_32_with_hits/len(llama_32_with)}")
print(f"LLama 3.2 Score: {llama_32_with_score/(len(llama_32_with))}")

print("*"*50)
print("LLaMA 3.1 Results")
print(f"LLama 3.1 Hits: {llama_31_with_hits/len(llama_31_with)}")
print(f"LLama 3.1 Score: {llama_31_with_score/(len(llama_31_with))}")

print("*"*50)
print("Qwen Results")
print(f"Qwen Hits: {qwen_with_hits/len(qwen_with)}")
print(f"Qwen Score: {qwen_with_score/(len(qwen_with))}")

**************************************************
LLaMA 3.2 Results
LLama 3.2 Hits: 0.310126582278481
LLama 3.2 Score: 0.22521097046413505
**************************************************
LLaMA 3.1 Results
LLama 3.1 Hits: 0.8037974683544303
LLama 3.1 Score: 0.6565400843881853
**************************************************
Qwen Results
Qwen Hits: 0.7341772151898734
Qwen Score: 0.6190928270042192
